In [ ]:
import yaml

def _w(name: str, default: str):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)

_w("env", "dev")
_w("tenant_id", "tenant_demo")
_w("site_id", "site_demo")
_w("config_file", "configs/tenants/tenant_demo/dev.yml")
_w("run_minutes", "5")
_w("event_type_filter", "")

env = (dbutils.widgets.get("env") or "dev").strip()
tenant_id_widget = (dbutils.widgets.get("tenant_id") or "").strip()
site_id_widget = (dbutils.widgets.get("site_id") or "").strip()
config_file = (dbutils.widgets.get("config_file") or "").strip()
event_type_filter = (dbutils.widgets.get("event_type_filter") or "").strip()

def _parse_run_minutes(raw, default: int = 5) -> int:
    s = "" if raw is None else str(raw).strip()
    if s == "":
        return default
    try:
        return max(1, int(float(s)))
    except Exception:
        return default

run_minutes = _parse_run_minutes(dbutils.widgets.get("run_minutes"), default=5)

print(f"[runner params] env={env} tenant_id(widget)={tenant_id_widget} site_id(widget)={site_id_widget} run_minutes(widget/default)={run_minutes}")
print(f"[runner params] config_file={config_file} event_type_filter={event_type_filter}")

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()

bundle_ws_root = nb_ws_path.split("/src/")[0]
bundle_local_root = "/Workspace" + bundle_ws_root
config_local_path = f"{bundle_local_root}/{config_file}"

print(f"[runner] notebookPath={nb_ws_path}")
print(f"[runner] bundle_ws_root={bundle_ws_root}")
print(f"[runner] config_local_path={config_local_path}")

with open(config_local_path, "r") as f:
    cfg = yaml.safe_load(f) or {}

tenant_cfg  = cfg.get("tenant", {}) or {}
ing_cfg     = cfg.get("ingestion", {}) or {}
storage_cfg = cfg.get("storage", {}) or {}
events_cfg  = cfg.get("events", {}) or {}
runtime_cfg = cfg.get("runtime", {}) or {}

yaml_minutes = runtime_cfg.get("run_minutes", 5)
run_minutes = _parse_run_minutes(dbutils.widgets.get("run_minutes"), default=_parse_run_minutes(yaml_minutes, default=5))
print(f"[runner params] final_run_minutes={run_minutes}")

tenant_id = tenant_cfg.get("tenant_id") or tenant_id_widget
site_id   = tenant_cfg.get("site_id_default") or site_id_widget

allowed_event_types = events_cfg.get("allowed_event_types") or []
if not allowed_event_types:
    raise Exception("Config error: events.allowed_event_types is empty")

base_path = storage_cfg.get("base_path")
if not base_path:
    raise Exception("Config error: storage.base_path is missing")

print(f"[runner config] tenant_id={tenant_id} site_id={site_id}")
print(f"[runner config] base_path={base_path}")
print(f"[runner config] allowed_event_types={allowed_event_types}")

In [ ]:
%run "/Workspace/Users/info@justaboutdata.com/.bundle/streaming_platform_engine/dev/files/src/silver_collector"

In [ ]:
queries = run_silver_collector(
    env=env,
    tenant_id=tenant_id,
    site_id=site_id,
    base_path=base_path,
    allowed_event_types=allowed_event_types,
    schemas_base_path=f"{bundle_local_root}/schemas/event_types",
    run_minutes=run_minutes,
    event_type_filter=event_type_filter,
)

print("[runner] silver collector started queries:", list(queries.keys()))